# Particula Primordial — Colab MIDI Lab

A tonic-independent starter workflow for **MIDI inspection, timing normalization, 4/4 standardization and transposable scale MIDI generation**.

The tonic and scale are configuration variables. The same workflow can be used with any selected key and interval recipe. It preserves input pitches during temporal normalization and does not attempt automatic pitch correction. The broader Streamlit dashboard and reusable Python modules are in the repository.

## Workflow
1. Install MIDI dependencies.
2. Upload a Standard MIDI File.
3. Inspect the note events and select a source track.
4. Quantize onsets to a beat grid, apply a target tempo and uniform duration, and write a 4/4 MIDI.
5. Generate a transposed scale catalogue for the chosen tonic.
6. Download a ZIP of MIDI outputs and CSV audit.


In [ ]:
%pip -q install mido pandas

In [ ]:
from pathlib import Path
from collections import defaultdict, deque
import io, json, zipfile, re
import mido
import pandas as pd
from google.colab import files

OUTPUT = Path('/content/particula_primordial_outputs')
OUTPUT.mkdir(parents=True, exist_ok=True)


## 1. Configuration

Change these values before running the generation cells. `GRID_BEATS=1.0` means quarter-note onsets; `0.5` means eighth-note grid; `0.25` means 16th-note grid. The output is a constant-tempo 4/4 MIDI file.


In [ ]:
# --- USER CONFIGURATION ---
TONIC = 'C'                    # e.g. C, C#, Db, D, Eb, F#, Bb
SCALE_NAME = 'Natural Minor'   # catalogue exercise label
TARGET_BPM = 90
GRID_BEATS = 1.0               # 1=quarter, 0.5=eighth, 0.25=16th, 0.125=32nd
NOTE_DURATION_BEATS = 0.92
SOURCE_TRACK_INDEX = 0        # zero-based MIDI track index

NOTE_TO_PC = {'C':0,'B#':0,'C#':1,'Db':1,'D':2,'D#':3,'Eb':3,'E':4,'Fb':4,'E#':5,'F':5,
              'F#':6,'Gb':6,'G':7,'G#':8,'Ab':8,'A':9,'A#':10,'Bb':10,'B':11,'Cb':11}
assert TONIC in NOTE_TO_PC, 'Use a supported note name for TONIC.'
assert TARGET_BPM > 0 and GRID_BEATS > 0 and NOTE_DURATION_BEATS > 0
print('Tonic:', TONIC, '| Scale label:', SCALE_NAME, '| Target:', TARGET_BPM, 'BPM, 4/4')

## 2. Upload and inspect the source MIDI


In [ ]:
uploaded = files.upload()
if not uploaded:
    raise RuntimeError('Upload one .mid or .midi file to continue.')
source_name = next(iter(uploaded))
if Path(source_name).suffix.lower() not in ('.mid', '.midi'):
    raise ValueError('Please upload a Standard MIDI File (.mid / .midi).')
source_path = OUTPUT / source_name
source_path.write_bytes(uploaded[source_name])
source_midi = mido.MidiFile(str(source_path))
print('File:', source_name, '| Tracks:', len(source_midi.tracks), '| PPQ:', source_midi.ticks_per_beat)

def extract_track_notes(track, ppq):
    absolute_tick = 0
    pending = defaultdict(deque)
    notes = []
    for msg in track:
        absolute_tick += msg.time
        if not msg.is_meta and msg.type == 'note_on' and msg.velocity > 0:
            pending[(msg.channel, msg.note)].append((absolute_tick, msg.velocity))
        elif not msg.is_meta and (msg.type == 'note_off' or (msg.type == 'note_on' and msg.velocity == 0)):
            key = (msg.channel, msg.note)
            if pending[key]:
                start, velocity = pending[key].popleft()
                notes.append({'start_tick':start, 'end_tick':max(start+1, absolute_tick), 'pitch':msg.note,
                              'velocity':velocity, 'channel':msg.channel})
    for (channel, pitch), queue in pending.items():
        while queue:
            start, velocity = queue.popleft()
            notes.append({'start_tick':start, 'end_tick':max(start+ppq, absolute_tick), 'pitch':pitch,
                          'velocity':velocity, 'channel':channel})
    return sorted(notes, key=lambda row:(row['start_tick'], row['pitch']))

track_rows = []
for i, track in enumerate(source_midi.tracks):
    name = next((m.name for m in track if m.is_meta and m.type == 'track_name'), f'Track {i+1}')
    row_notes = extract_track_notes(track, source_midi.ticks_per_beat)
    track_rows.append({'index':i, 'name':name, 'note_count':len(row_notes)})
pd.DataFrame(track_rows)

## 3. Normalize timing to 4/4

The normalizer preserves pitch, velocity and channel. It changes note starts and durations only. If your file contains several note-bearing tracks, change `SOURCE_TRACK_INDEX` or adapt this cell to process each track independently.


In [ ]:
note_tracks = [(i, t) for i, t in enumerate(source_midi.tracks) if extract_track_notes(t, source_midi.ticks_per_beat)]
selected = next(((i, t) for i, t in note_tracks if i == SOURCE_TRACK_INDEX), None)
if selected is None:
    raise ValueError(f'SOURCE_TRACK_INDEX={SOURCE_TRACK_INDEX} has no notes. Available indexes: {[i for i,_ in note_tracks]}')
source_index, source_track = selected
notes = extract_track_notes(source_track, source_midi.ticks_per_beat)
ppq = source_midi.ticks_per_beat
snapped = [round((n['start_tick']/ppq)/GRID_BEATS)*GRID_BEATS for n in notes]
origin = min(snapped)
out = mido.MidiFile(type=0, ticks_per_beat=ppq)
track = mido.MidiTrack(); out.tracks.append(track)
track_name = next((m.name for m in source_track if m.is_meta and m.type == 'track_name'), f'Track {source_index+1}')
track.append(mido.MetaMessage('track_name', name=f'{track_name} - normalized', time=0))
track.append(mido.MetaMessage('set_tempo', tempo=mido.bpm2tempo(TARGET_BPM), time=0))
track.append(mido.MetaMessage('time_signature', numerator=4, denominator=4, time=0))
events = []
for n, q_start in zip(notes, snapped):
    start_beats = max(0.0, q_start-origin)
    start_tick = round(start_beats*ppq)
    end_tick = max(start_tick+1, round((start_beats+NOTE_DURATION_BEATS)*ppq))
    events.append((start_tick, 1, mido.Message('note_on', note=n['pitch'], velocity=n['velocity'], channel=n['channel'], time=0)))
    events.append((end_tick, 0, mido.Message('note_off', note=n['pitch'], velocity=0, channel=n['channel'], time=0)))
previous = 0
for tick, priority, msg in sorted(events, key=lambda x:(x[0], x[1])):
    msg.time = max(0, tick-previous)
    track.append(msg); previous = tick
track.append(mido.MetaMessage('end_of_track', time=1))
normalized_path = OUTPUT / 'normalized_4_4.mid'
out.save(normalized_path)

audit = pd.DataFrame([{'pitch':n['pitch'], 'velocity':n['velocity'], 'channel':n['channel'],
                        'start_beats':max(0.0, q-origin), 'duration_beats':NOTE_DURATION_BEATS}
                       for n,q in zip(notes, snapped)])
audit.to_csv(OUTPUT/'normalized_note_audit.csv', index=False)
print('Normalized notes:', len(notes), '| PPQ:', ppq, '| BPM:', TARGET_BPM, '| Meter: 4/4')
audit.head(20)

## 4. Generate a scale catalogue in any tonic

The following interval recipes are independent of tonic. The list is a practical Western 12-TET starter collection, not a complete census of every world's scale systems. Add new interval vectors to `SCALE_RECIPES` to expand it.


In [ ]:
SCALE_RECIPES = {
 'Major (Ionian)':(0,2,4,5,7,9,11), 'Natural Minor (Aeolian)':(0,2,3,5,7,8,10),
 'Harmonic Minor':(0,2,3,5,7,8,11), 'Melodic Minor (ascending)':(0,2,3,5,7,9,11),
 'Dorian':(0,2,3,5,7,9,10), 'Phrygian':(0,1,3,5,7,8,10), 'Lydian':(0,2,4,6,7,9,11),
 'Mixolydian':(0,2,4,5,7,9,10), 'Locrian':(0,1,3,5,6,8,10),
 'Major Pentatonic':(0,2,4,7,9), 'Minor Pentatonic':(0,3,5,7,10),
 'Major Blues':(0,2,3,4,7,9), 'Minor Blues':(0,3,5,6,7,10),
 'Whole Tone':(0,2,4,6,8,10), 'Diminished (whole-half)':(0,2,3,5,6,8,9,11),
 'Diminished (half-whole)':(0,1,3,4,6,7,9,10), 'Double Harmonic Major':(0,1,4,5,7,8,11),
 'Hungarian Minor':(0,2,3,6,7,8,11), 'Neapolitan Minor':(0,1,3,5,7,8,11),
 'Phrygian Dominant':(0,1,4,5,7,8,10), 'Chromatic':tuple(range(12))
}
OCTAVES = 2
START_OCTAVE = 4

def create_scale_midi(tonic, intervals, bpm=90, octaves=2, start_octave=4):
    tonic_pc = NOTE_TO_PC[tonic]
    base = 12*(start_octave+1) + tonic_pc
    ascending = [base + 12*o + step for o in range(octaves) for step in intervals] + [base+12*octaves]
    sequence = ascending + list(reversed(ascending[:-1]))
    mid = mido.MidiFile(ticks_per_beat=480); tr = mido.MidiTrack(); mid.tracks.append(tr)
    tr.append(mido.MetaMessage('track_name', name=f'{tonic} scale exercise', time=0))
    tr.append(mido.MetaMessage('set_tempo', tempo=mido.bpm2tempo(bpm), time=0))
    tr.append(mido.MetaMessage('time_signature', numerator=4, denominator=4, time=0))
    ev=[]
    for i,pitch in enumerate(sequence):
        start=i*480; end=start+408
        ev.append((start,1,mido.Message('note_on',note=pitch,velocity=82,time=0)))
        ev.append((end,0,mido.Message('note_off',note=pitch,velocity=0,time=0)))
    prev=0
    for tick,priority,msg in sorted(ev,key=lambda x:(x[0],x[1])):
        msg.time=tick-prev; tr.append(msg); prev=tick
    tr.append(mido.MetaMessage('end_of_track',time=1))
    return mid

scale_dir = OUTPUT / 'scale_catalogue'; scale_dir.mkdir(exist_ok=True)
manifest=[]
for name, intervals in SCALE_RECIPES.items():
    safe = re.sub(r'[^A-Za-z0-9]+','_',f'{TONIC}_{name}').strip('_').lower()
    filename = safe+'.mid'
    create_scale_midi(TONIC, intervals, TARGET_BPM, OCTAVES, START_OCTAVE).save(scale_dir/filename)
    manifest.append({'tonic':TONIC,'scale':name,'intervals':','.join(map(str,intervals)),'filename':filename})
pd.DataFrame(manifest).to_csv(scale_dir/'scale_catalogue_manifest.csv',index=False)
pd.DataFrame(manifest)

## 5. Package outputs


In [ ]:
zip_path = OUTPUT / 'particula_primordial_colab_outputs.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for path in OUTPUT.rglob('*'):
        if path.is_file() and path != zip_path:
            zf.write(path, path.relative_to(OUTPUT))
print('Created:', zip_path)
files.download(str(zip_path))

## Interpretation and limitations

- MIDI note positions are stored in ticks. The file's PPQ and tempo define their timing interpretation.
- This notebook standardizes note onsets and lengths; it does not correct scale membership or compose new note pitches into the source performance.
- The scale catalogue is an extensible list of 12-TET semitone recipes, not an exhaustive list of scales.
- The repository's Streamlit composition module additionally implements seven progression templates, mathematical melody contours, extended chord stacks and an open-ended ascension study.
- For production instrument rendering, export the MIDI to REAPER or another DAW.
